# 📋 MNT02 미션 03 — 이동평균 및 지수평활법 기반 1차 부품 소요량 예측 Baseline 알고리즘 구현

**Student 학습용** / CMS 편성 3시간 / Level 3

---

## 시작 전 꼭 읽으세요

이번 활동은 **교육용 가상 부품 데이터**를 분석하는 실습입니다. 코딩이 처음이어도 안내와 TODO를 따라 한 단계씩 진행하세요.

1. 아래 **준비 셀**을 실행합니다. 필요한 라이브러리는 실습 환경에 설치되어 있어야 합니다.
2. 미션의 입력 자료·핵심 함수를 확인합니다. 학생용 `TODO` 부분을 직접 작성하거나 허용된 AI에 도움을 요청합니다.
3. 코드를 실행하고 실제 출력·오류를 기록합니다. 실행이 실패했다면 실패 내용을 그대로 남기고 수정 후 다시 실행하세요.
4. AI에게 보낸 **실제 질문과 받은 답변 원문**을 작성 기록에 붙여넣고, 코드·출력과 비교한 수정 이유를 작성합니다.
5. 마지막 기록 셀을 실행해서 JSON과 검토용 ZIP을 만들고 내려받습니다. 파일 생성만으로 학습 PASS나 사람의 승인이 되지 않습니다.

**인터넷/폐쇄망:** Colab과 외부 AI는 접속이 허용된 환경에서만 사용합니다.

폐쇄망에서는 사전 설치된 로컬 Jupyter 등 허용된 환경이 필요하며, 외부 AI 사용이 불가능하면 '미실시'로 기록하세요.

**보안:** 실제 부대·조달·업체·정비 자료를 입력하거나 AI로 실제 발주를 승인하지 마세요.

**실습 전 필수:** ① 환경 설정 셀만 실행하면 됩니다.

실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.

준비 완료 문구를 확인한 뒤 다음 셀로 진행하세요.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'MNT02_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
MNT02_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global MNT02_PROJECT_READY, MNT02_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        members = [m for m in archive.infolist() if not m.is_dir()]
        if not members:
            raise ValueError('ZIP 안에 실습자료 파일이 없습니다.')
        if len(members) > 15000 or sum(m.file_size for m in members) > 800*1024*1024:
            raise ValueError('ZIP이 너무 크거나 파일 개수가 너무 많습니다.')
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    MNT02_PROJECT_DIR = WORKSPACE
    MNT02_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 4개: MNT02_Colab_실행안내.md, MNT02_M03_forms.md, MNT02_M03_lesson.md, MNT02_공통_작성안내.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAgIAAAAIVz+D1bE0QYAANcMAAAbAAAATU5UMDJfQ29sYWJf7Iuk7ZaJ7JWI64K0Lm1kbVdNTxtXFN3zK55UdVkH8lW6RCmqLAFFcbLpBpngSG7B'
    'tmyzSFcGBuRgU0ywwziMydCYzzrN4A88qGbjnzPvzX/oue8920PCAhkzb+4799xzz718x2bnXow/ZM+Sy9FFJgp1//1bJip5vt4eG5sIqacLmezqUiyRDcVT'
    'bxKL/KTHxEHba9lM1HP8xmBiMycOSsKw9HGEEdsdv2zwY0vYNX/bDf0ZT4kaTpoNfnrLC3leqIeYsA3RbAvbEkd7vOWy38LzXqvHvE6DXxt+5WJwznMdcdxm'
    'Im8Ku8L83TbfKfPtMhNHDYXYr7T5rsH4zr/ifVvsO+IshyS2AUKHGHsYYn7FEB/KfsVivNYTtTZ3i/wj4DT4WZEJoy5uzMCdidWV1Jt+NxVNLEUz/W7mVZy+'
    '4uOPePaH5Vg0neh3V6LZ1HIyuxxflLmtN8SHi0AIUXX5dY5IRYoA5+BS33D63an5MPPX6/2uODqR/B04XpOeDjGa32bwKMT4F1dsmmwmlskkcf3rZHolI2om'
    'EeabBvNahqjcSii1WyoPOBU1l04QNttgfrHEC+cAEEa9GD879lwXh3vyHbsSxK5AT4UZ8Ijm536XFzq8lQPm5j9MtPPQSL/rdXvelcN4MSeOkRdvtUXVwJGO'
    'Jf7eBKe2v1Gj2PTguD0KP/Z4kAxvGVoGbCn2mqQVX0ktx1agtmg2nkww3nDFUQl6I8TQiTCuUG2WjqWWo69iC69XE6/onLB6EBtC5cBpMA/U+ZQg1SgMELsO'
    'WPGcd6IGgi0bPN8jtNsy37dIYaHFGFiOUWiVAy+ZUvhHJr/8j5cswsebFVxgWPxyk6KCEZSdl4ZR55LZ8CCp2NJ0Op1MM7Gx5m9YqmuklKWOSQKivocajV4f'
    'e0LS7YhCAx/n6IFBuf1Dg6VXEwsr8UyGGKAaNk1RK7JhUzDICh1FtH5dAmKGIgxSvi57Tg6ncl7zFkrqofqSUEkF0gNPe/jEU8gsGPfG8DqlQFzUVQMkGMVz'
    'aRfGFd8+kXfVc0S+rtY9naq7XLapUsh1Dk0T7PUhM09DLLmaTa1mMw9meTPvm+4D/0NROOeklUx2CQ/RstmlWDpNyAmySqLr8tNevwv2Qr9niDoQAKUC5ahq'
    'fHcTVsX45Tnfz8u6OjlUDUBK/FORsvKcNXzRb6Jn+ZeczKpypKHLzpME6qzkRXh/t6oZkNHtwXFSuL0GE4HKQIzUxR1uvFbH3+roKsmCHzTgYQOQw/cJzEV7'
    'UE3eLAsLPcCbBrVaTRadH9cCTP4Ip/+RibM1Ye8B4Sa3TwKuPsjcrqDa6C8UHDd/9is2cnjkXbVJTHTCa+b8LTsoMviOZUM+Zl6swyFQIr9qgi6mrINo9Kvg'
    'xQVkaqyZ6annc9PPF8Jz8y9f4OISImi49xkJo8AFi0XCsy9npl5M/0zKhFSk6FQ2yvOG/mCJa+s+fx04nnJxWVdYxmkNtDmW+ETh9mSH4r2DrYAvT4K6SVT6'
    '0fj491Qxf/ctENEfuOMQlXeGnbTZagXTSlQhKdVNhKrqjh5JcekhKFmBPnihQXer3yiVr2vIqFEqEJx7LJVGzdyhjPx3eX55IecEBgRVAanQr7DGC2eY9+EN'
    'KlEm2XjOlSjYw0DAXDF4oTpI+CckPDFO8Uh2O672eSo++JIXq9ZRYwB8QD9kLnfCYK60hbkmtQUg+9bEuOcAxtGJ7Bo9M1su/BWAMSDJdbqW17yQcwaskHNS'
    'mfVMGEni2oJ80UJUDcrcOVfLAp3DUDbQvt/CoTlDUu80MMpkLutSReg3sg29YKy3IQ5qMtyHaIplgx4p6Y/qPTYxLueJnMG5iXE9uwcqyJskpoA6Dku0AWkq'
    'Zx/3u7NP8TOpJjPJQ/d0ZHpm+hmEvjAbjkTCv85FSEcUXEbUDScFpPUhBQSnF1s72rugbBNdJPEbw/Yb8vdsNiL3kLz5QMPRpaIdaaAsmbN6NZgzNkZ+lifv'
    'OnsrDQSH9fBSQ4EGJcWUo1NTopZOhZfEKdtW8UnmAr+WNgDRw7gpXb9YxNLCKAfV6ncsZBU7WXQJI4F20cHiiR2P738OzL5BidXs1bsOFhn0QEBgENMFHiNZ'
    'eFelSlutlDObn4pEAKxT8tpQ0vaN3LMM7Rn3+4uepMPYtIbJujfkq8F1We+4cknTjge63tVJgNyuc+cwYEBjSjtgbiPPfolnmf+Xq7X5cDL0BGbkOUZAaqiv'
    'bLGtIi0T6rFenOQ8uZaDh+9foRxyyfhGIQPLQAQshyr+SBXKN4KCkv9b8I+Ov4FLDNoeUWwQLbtBDi21tanVSgXX/5CgxzFytrErS1lqgxo5TdmRtn5m3N0M'
    '/gdQSwMEFAAAAAgAAAAhXJA3ewbaBwAAHhIAABIAAABNTlQwMl9NMDNfZm9ybXMubWTdWE1rG1cU3etXXMimBdtykqYUUyhOm0WgCYUWuigluK1LQ+MPrKRg'
    '8GIcj4NsqVhJNM7ImXHlVo7tVCFjWbbGoGThn5Ls5r35Dz33vjeSnDZpF/2AEmNLM+/jvnPPPfe8nKErVz8bPUdXRs+T3ryr3X3S63d1KaQXTpV02FZrtXRt'
    'JenUSUVrpHccXfTxIK0FquVREkcq8umsjnZJHTnpvSLpO2W9UVU/rZD2izreoosThckb16cnSXvl5KCuHjX1I5+Sw2bqu7mcflhRBzHxHpEzRl+knosdvuzN'
    'Oulc+micJBr+gI3S1Zje0z8vc3R6aXGIdFDXoa+exqkXEJ4kbYfSale3fLqQRIEd99bs+6MjoxfeHuoHpKI9nFQduAi1IUc4dpPDiq45NH7xQ15KBy7hUOrx'
    'HunGXVKlXdk4cNMHvn2hQ0yvB7oUpOtFtVrFMUZyOXXgJK1nlBx08W6MPr30KU3cmP1uQm13aXTkXR101VaAAKrJ8y4lrYgHYlt9GOh2kdKHrjpok3br6VJo'
    'X/NME1/q+cBxEIqTjjl20nJ03QMYlHpt7YYcGz97FAIaVWogsDNnTkObyy0wDkgmFpEVTzoWvAWsUtOrh3mTHOQ6iVw8NWCddGaT6N5JRx256vEuP5ZMYpRu'
    'ODZmWYIDwZpekADoBWw3PDxMr/2N93Tq5796wEiNX2aGpxs/nnRAJBUF+CtwCvG3wlxumPTtpq6DJeUK2NGroVKg65UxerG+xYuo0iEIQcgp5hEP5xf2qX7Q'
    'VptlzrmdjUXHL+sHFVI7W0kcY8CyaraFHmByUB/jXeuA8zD1QIPNivqljOQ9q6r7CJC56D1j8JEQUAWUtNngeciQDvHTlupcWuSSGaAfim9nkZBPtd3miThc'
    '6j0ZE5KUnbRWJWZ8fZHTXWpgK0Kt69YTSX4zJuVXZFrd0ZvbXCf62O+PzQ78xrFZhD6pvaow6/WUyb0hh5AWCQ0MlW116PRilgC4OJARvdVOvT1VKnJ9kEGR'
    'bPnaELOc9+sw8rUv2PECNdeioNZcS4z+igwdGKDdhnq83Duz2cXW90I/h6VGur6CYlku69UGV3fS6Sb70SlYDNMkwUIHng8F3HEwv+ij/nHig7baFuChJxAs'
    'Ey9mlnXYxTCL9T9Ri7nzyX7bSq4gXHb0VjSIsJyVVKuY+jGDqaPnBuQl5nzSiREbTgP2V1jmICCqdEzqdltv7HFDuu1neYSg6NtRBttAfsBQ6NZABkw+hezI'
    'z+o2IC6XVdjlpuEaeCXhXJpZIamWy19776RAX32H3/J8MIIB3hv4bZ6zdP0NsLM42c5pBFe2tbtBAI4CPrXRbSD4hBvYQja0r9qQs56oW0jlBHje412pCUnC'
    'hKcxPjJuFVAMzb7ISy9kBfDv6DoLn+mQPbUzX7nC2kXtFW1F2xSIUmYpW45RUOpoUcoDzNIuy3knVo9wXnRjfLTYDcwGgXQItj6N9bKfdW3XZ1nG+aXY+uKK'
    '3szeA3KR3qmzfUiXAtMDUAWs/PJxUDHkiV49Rmm/dKqqxFRHgeBcYRc9JMvDS8eTlKfeoS41rXfq9SBusrsofRGSZVUHu3EI/KW0BjMQ8/PWr5SJixEJGqBt'
    'X3ujIGk1xUb5DShIb3BGhzdKcWGygG/yDwboaWzD5E3KuyJ77j5qr09DaUOeq0q1njhwnO4+WpuNTXh9xKbqVBnKO5EX64j6csswwZyu1cTX7bhy5le69oId'
    'wv7EQGRtS78O5MB9qbTZFO78Cd/x5iw+s6OGQME8jJ3y0XKi0zba8uovuWg59E5Z3W+IOC5V9NIKN2QUNBs+sxJnbaMqHAtj3jAjTtuB/R40rqK9DBxrL3vv'
    'jaqk3/LIyAO30F0gAThgdjb2mPqcSONyeH2kHq0JsyzxWJm9Z7KDND5rWXtNzJgm08lOOnA7gHnTzfJkYBPzarppZqw/sOxaoHMZwsKxMSrMfZ2fmJ64MV+4'
    'XshPTd8cPXdt8puJkdl57piIDl3BGA0TiYkPJWeRUlEEG0bzQ2LS6cWdeySOuQl+bop7t2kwA0dYbQSbe2IrwFpAi/eG1OLOw7YwVabJIKg/dxfBjKMyM5BL'
    'Mqhaiw5KF/OgNVoFnuShPlzaJk72aKpV5VuJitbtjYJ7537EvLELmNgYGuROtpa6M90AeYbwidDVYjsD8gKw9dGe3tq3jhFHMnWYqYYRSOj/3SBvRBNTrH3q'
    'p8ec+uPJQmFmmtMslWsKl3eQ2jKCjDisi/p9cs9nyRU2/F8uhVZaeiVlOzDcERyTUdpjHzCbDptd8qS/5Y0M92TPql4fuSvjn1yStBmEDalHsygk0WQ2HKLP'
    'T40Fz/dQDSywGA9r65Wv5sfZRCHTHP+AczN3zIPu7PAPEzduTcoioi88+7SQM9OWFjEl5OsQqIhj4syRryDtBqO8flhlZWkIiHKi7B6rKoFNlrnL8tVRfJ0V'
    'k15I3Pd/6aKs88ZjsfnEqY16M2WlFWeqjeU3mwbEP2DdO5Z1GAiDOkbTMzcnv5qZ+X5AT65lEjNyfXZ++qshmpucnZm7OTiicGtqamJufmTqmyH6dmZuqpCX'
    '/2C5hpWvyXe84cowiZL7nnExPy8bYybXLFNYpvngtZzN2Ecp4QieQ1X87ASZN2OXArGRGwDuN27AD7ItzIZiNYwbEUn00P/SzaLN9aug/AZQSwMEFAAAAAgA'
    'AAAhXMNDYruPCQAAVRgAABMAAABNTlQwMl9NMDNfbGVzc29uLm1k7VhdbxvHFX3nrxjAKJAANKmPODWEAoaSCIWBKgiaAgFaFBZts4gQfUFUCviNslYGLdIV'
    'ZXOlpbRkVw1lSy6LrCRKpAAqD/wp8dvO7H/ouXdmuUsncfPapg8WpPXszP0459wze0PMfvqHsQkxOzYpvi/WhGp05HY93H4adD0h/W2hXhVVycGDsO7KM1sE'
    'PV/6jhhX/rGQl8XweUmoJxW1X5N/fyqUU1K9Q/FRrpBfmF/KC2VXgnNPvmyrl44ILtqhY6VSN24ItbEe1k+C876Qr0/CHTeVCvwiHkY7ymc+AgktHw8sZTWE'
    'LHdk+Uo1q/LQVZYrEBKOHHRl+Vj9Y5P323AHXdVs4zwRdPvBqS+P+ub10D6R5ZIstzIj+dH5b6Wn3D5O+HnZqYYl5KuKfNGiM8ONqtp4qrx1c6bZSQcq8E81'
    'enRgaF+oclsEnSLKInAAKkA7Kf87uVWjGHWFDqryvEevh7tPhTyrKddS9aKgyP1iKnUzWhHaFhYJ+W1PbTrIz7PUWUeockttXai9KoqFbc0+cR2G76NmGyXx'
    '2/k1Ef6tp6xTMZG5pcpu4FtvijXapXcY+M9RNfTSkmXnTdGOz9a7TonJQffDwHfVwSh+0oLaU3JHipwWqrkjNy0R+HXVKGJJGzg7UeWGajhidnpm0P1i+jP8'
    'nMVPaqHOmNCDoIQ5R2cm5JUVXFRD29W93e+o/RMd/aAbHpSkd4xQARNsgKjkESD94pTbzm2hEwEd9CI4pyoOuis3/5pb+DpP5wZnReXZoe1Qd2c+mUaHjsOt'
    'nrgNwA1b7/HB0x99/F5YqUivFD7x3n8LABvraJ2YGPtVstnlhjy3zN5Bt6daO0JDOYI0sMaoGr4CliAzkypa8Cfd+D8P4TjoUpBM4reipaYgDFTe9ZAyoIJt'
    'TOIirPXVmSNuoa5m3XsrvxnLjN16Px3zWDcIIYMDLeYGF54AidyjHHWwIk4GB1rhnpNM3OMsdksAJZqSSWDpsa8uXdnuT4ml5bX8/eXlrwrZxaW1sYl7+Ye5'
    'e7ml3MKjwnwhM7/yaOk+KxNniSba1wJUCrfbyFdzx3OD8wtBavFNhaiCv8In9GCHEK7sknzcGXQ/Xl7I3SemEMOih5o4Qm72sJu8XI+ehDWLpMdrUFH/ePcz'
    'jvy6Jl+4U2KusPogGwUYx5xZeTSXifge2mCCw7hiGcS+XlE1jwwG1Nk/o0jCA0tM36XSqkuqkZDnxeDseoS7Q32cEiOBzT3MreWyv5+Z/mR2JrP4cI4l8LiP'
    'o9Dba6Rhtpj8NdCD2FH1NhRj0B2/HT04qKJBpnaD7sQ44UK+PpbP0KLmpvSOqI3IASAflZOyi/oKtbsDpKAmPFbuYazc+8vy6mKBouGuPVx+UMiu5leWV9cK'
    'xD8SrrrFfdHtCcuAhe4jtOSsZKSN1kbaaQbJTQHSqUb/nR2gVVx7WpUvzDEcLIBmKtJm6ftqryMepXMLK1/mxPdPngumWRugaIrPZz6PhF8vhM50SpiCLItF'
    '7hx4NNRJ7iZ6c2CZ13gRiOVZXCOvSpnoNyAh3IH9k0jC5FUpC26BqHiSVY0KabsRKyBCzwEEsis4WJJ2mnSYVJEGcmxGLflo61RuHWkRNapJJKz3YinBPJfg'
    'Szx01UaDO6ExS+PDMIoBTzAw03R2nJJ58GX+wVfxMgNpCOk84JiPMT+Cl5FeMuiZA9YpH8RUYJU976jDTqSUPR9/jPIC3QS+XTolGnIdmoK0YzQe4ukPLnsu'
    '6CVkpRjWyUCc9zCYjWMgzYeCg5tOSx7pwns2d8/1mEuPj4Or9SgJwmho10kvyD1s1+nJDfP7OCsxoBbuP+OJw0wDYqvQxlTKqJ62PxhQcFUkglxg8jFEP86p'
    'cc2I2u3I3RbyslEQ6mQECa3iPJ+bNDEMVA3S9qpkwaSHrIm+JVQ0uKrc4bHKgSRqA83BcvnqMOj1DNCiQEisXFAGQ7VuU7kwD8gssaDbibYm8p/g/I2QcVqm'
    'YanUT0jiD6UwOPNROi68gQE2ed4yu6JmqK2RI7zP2XrrzKZhViMgmWI2a+Zg07HMh2ZIS78WfNc353HEFy6YRLDEyUClF240kuEYYOmA40lLbortBHsHLVkd'
    '8gIUJJ69bESsS1RKG2+DQOBSHfocQrOtywCwGBxyGRkVI7MjWpgEuhnTlxa0e9C9rWMcdGlaB72Ssg+HXofQpGlB89c7ogo0+urKwfZ6St8hwiirJV9TgrqX'
    'uiSTwWnHuAYGQ4QWDoftPadjAqUm1sHK8gW6IrfptxYNGw2qH4fRB/pWgnbATmjMajYZtzBkUmQTIRGtIvGa+66FIIpaD/O3BYAkm2wy19XFtkn4fNvDIMAq'
    'zCBIuWcPt1BbV8QEffMwIB6ZiYINLKkiv2CGzVjkgliATYPS4ouRtUANLkcW5YP1qKJd+TQ7zaw1LjButLGv/ci50iZ806C32bPXo7XmjmXUHfGiNUhNq7/2'
    'aFm2y9QXMnGM8gjTsuoas6hxDam3uGYMjNHuGdhD1xj0Qn0DTAHD1VTql1wVDS/Gb90CUwxaycnM/WfTS9ZlzjinxKLC14uLudVHZLCwgM2WsVKgRTox8mhg'
    'nP0LgTd2QMG00Tj8nVSXdKS9kEVcTNDBbk++7NMESIzUpG/HSMLdWGdCN4RYRCP1YbNk8S0Mq9P0CqKjq7ImtNE+2hR78zhzorj1RE7QDEdAOki0aWYeJu/2'
    'dDUkSnJtuQnMUaPxEUgc3EYBga3YDRu3yRrDfsAx4kPpmIkXdWo8w59K9Fyf+m/4nEBjke6K+zW+9bGHj3tCFzUe9GxDiQxwi8Y+0P6wzMTc5qYeDsY78wkG'
    '4UbayUbRSrY5uq8YN3fps0XTioRAl42noW5edBG8k5owheVRPiV+0tBTdxAU2pscgjos+mTxf0/fyrwpkjdTlyfq8FRPcUpJe2tNbzTe8Bu0yGqO4xVjs4Zd'
    'MVn/Ll8oLC9Rd9l569lKJ2gN17eCvSo0UznriZ5Omp5y76f+Rz5ZGGUaEsiIHKYGzJBW0ysH1QUH4u9HWmuzWmqHtxtjI+Ny/0KHI08nWITmUTb2lXRX0d9x'
    'gVRzL9MOkr/zGssbg+0DDTat/j/nI1JavGuYpvUozf7wOwbxQPeHpyyi4ckBqUCA7Co1jUj6rhz8N6ekL5FMWB8jUlaH5vpHPKZs98iR4kF0hD6Q5xnOIPUk'
    'AbQxncJmybQ4rkUq8XH3nd92s3pLLXPDL7wZc9PWtybkq6eMfEzfV/VDUhJ9Sd2qEUdfFRNe599QSwMEFAAACAgAAAAhXLEHCSjoBgAAnw0AABwAAABNTlQw'
    'Ml/qs7XthrVf7J6R7ISx7JWI64K0Lm1kbVZdTxtHFH33rxgp75iQj6aPKKEtUkBRPhSpLxFNaYJEcQREVfu0hjUytlNM8MZrsussjY2BLupiG2wU8+Kf45n9'
    'Dz13ZtZePiQ+LO/OnXPPPffce4vNzD4fn2CD1mm4ccpEdVuYJ0xYWb7WTiQGgSHW00xkbb7v8zNDbHxg/EMg3HZoBnzPoSdityTsrOju9Tui6g9aHuNnpjDd'
    'fofnD8Q/GYYI4brDBk0j3PDwUu5cuF026CKAK1yTiXxN5E5Dy+a5Uiy6DBLaFgtLJq4SnhvmuqKa4fksz9fGmPhc5K2ucG0Wfszyo8N+Z/bZbL/zw7OHuAOw'
    'zm2JAClQ3MFpUTh7YveQqZzigXC/5wC1wQtGvzNomcK6QIxyRjTblFSdnwNltci/Fni9x+ioVwde0TDAVA7gdahE4tYtNjnNwoqDmyQ0ywQWsNsTnpVI3B5j'
    'i/MrK6klXNMNRA3nL0p8x5Fh3QtJHuWEnzaiWPgUWmB5zQdIQHFt7nf7HVwhHFNUTcabJXxCtKYh9l3Gg0PetIhUfoAbEd5Lj+Axnj/lLSP2mLdrwunxQ1y2'
    '7Yi8E1qHUSoTYyyCmK+FnzZRvwBpEFLeaou9NgggvJPTolzU7PQ7oQ1Yvs4KgD1Hcfm5GNW3ZQyaF7Ls3uUbozqQnprHONP8NwYYaFGIGL47Y1J8nhWjMCwU'
    'hNsjPDywhZ0OrTaQ+SqB2E3E39ce4uEWeZgJ0wvXXXZncIITbpsE4ppAHeVFolgLkF+mIHI1EmKF1CL2ghiku4BUtfnRN15UcoKEoQGiq2LqvCT5wxKrdtNM'
    '6nRiRQIheWeogxFRjgfpKZV7FlTeavO6DUTi1KGGU8lLiZzbsmBrB4PzdATzXtQ7INXiXzZ1nRFPFUAVGmH/6/K8L/+ju+VN1DvyRrzpFhg9ydhMlAOcIcSq'
    'qWNQX779M/lT6o/ky7dzqyi/ypxeC3c/jC4MHB44yaFBqCzCQpEMBARC2YOTYNSx5EtpXXZJcOQZsBoXfAoXyt7K8P0LrV0pIMekVho01/EuPww0eCpz1hGm'
    'g2qSy2iptOGHoBn9W0H/KAej5CGHrYrwTBDDiyTowakPEeCDsPBlZdhpUuYjZpXKeNPEWepW2Uq5kg5fLZJLSU41qnJRNNLC21bUxmuvKyf1OnO335m5j98H'
    'fIsiw6/IKsBFuk0hkBWAZe1L6pcmpRAoh2rDZxMJFRhQ17Psx4VVFv7d1VROPBi7Bx0OAvOa+zJyfnr0kXhEAt09+gxL2XM4GFzzb7JJ2KEnGzfvgBWkjj7I'
    'ik9tJMHRvihPt8C/+Bg5UiJkPmHFJtsoF3kpwDu4hB5JTHgdxcC5RpYaApFihnd0CGNjSsnIZdDpqjpfEyqio8cxJnzElN5U6QIjeo9QgUXJLht9Ve9R/+e9'
    'm+cAyIVkk9T6npVU1UqGW5uULO7GHSSAM5MfHVCko8wVK4RYyxSXJi/aRg1SurnukT2R1rIQMQYTBg36sWCQ1CQhLNw9UOrS01s1bax2Wml5X1uuHuJwNN00'
    'u20yzyjMsP9vnJkYviy0uygQ5EKaKme4D6Ia27LxDDXd9aDiwad4G0OIahvAOcn5mRMJceY72fmy30itsHFAeTY98+Lx5POpR1p4cUS6X45Dy9MjGmVCPa+0'
    'GcXyaERGY9x5PDX5dHbq6avp2ScvnkfaNenwVZFQHbIYFuoSjGKEFOU2CWu0XuDGM1N5TDTR0JNYB6oYk7D9fTeq4MgtoIIdoD7DhHAomspXFdWFF3mT06RB'
    'xQbNA8jwpsaCjWKwDS2UsDRqUh4QL1UKMqZKuaQIEk+1DuS6CDS9dR31sib79/hyLrKuQ8s3RHkjrnu9y8U2tp+nn4Dsh6nFuV/Ikmyfe7Xwo4GceKaL5/ws'
    'rbzj8ojGxMTgDT+bHG3dMHhjU2QM5Wnv3y2m5n5dScrFdeyvhXc4zxsFvnM8PAzEl6uuRiWKHW0mVDzTkdSElVLkM8MdBxWRJJEkud+L6WxhaWV1+f3r1dQy'
    'E0GXnEyPGAOv0gFIAmsa0v/mD8cPETiEg1qOLqUTkn1KImYo1/dKitJEBWVpo81XmjR1j5o0iUSc+qGFxxxcyhubPvakONtQpEfLCrUGqJLv6uO0wmURRK12'
    '6p5rCtaZbBTISORp4jEPaz4nQnSGhuqreNd6DiYt5o3E7eOv2q00tWrnzpVWUu+XX8+/Si0vvFlYmltcSb5ZWH21PP/b/PL8Er6fGJ+4f3t8/HvSxyX/J7LW'
    'sfgcPJ2afDQzhcAqi4Xf36WWV2no7pygxXVSV6ebWt61e1OB5d5Fi2zRvjptXeP2OO0x9d7VecuiL5xo47Dv4E01DNBemFFUe1jAdvHqqpf4H1BLAQIUAxQA'
    'AAgIAAAAIVz+D1bE0QYAANcMAAAbAAAAAAAAAAAAAACkAQAAAABNTlQwMl9Db2xhYl/si6TtlonslYjrgrQubWRQSwECFAMUAAAACAAAACFckDd7BtoHAAAe'
    'EgAAEgAAAAAAAAAAAAAApAEKBwAATU5UMDJfTTAzX2Zvcm1zLm1kUEsBAhQDFAAAAAgAAAAhXMNDYruPCQAAVRgAABMAAAAAAAAAAAAAAKQBFA8AAE1OVDAy'
    'X00wM19sZXNzb24ubWRQSwECFAMUAAAICAAAACFcsQcJKOgGAACfDQAAHAAAAAAAAAAAAAAApAHUGAAATU5UMDJf6rO17Ya1X+yekeyEseyViOuCtC5tZFBL'
    'BQYAAAAABAAEABQBAAD2HwAAAAA='
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 상황: 어떤 문제를 해결하나요?

**미션 개요**

이동평균법(3·6개월)과 단순 지수평활법(SES)을 활용하여 1차 부품 소요량 예측 Baseline 알고리즘을 구현하고, 탐색적 데이터 분석(EDA)과 ABC(파레토) 분류를 통해 수요 요인과 핵심 관리 품목을 도출하는 실습입니다.

**배경 및 필요성**

본격적인 머신러닝 모델 도입에 앞서, 전통적인 시계열 예측 기법을 Baseline으로 삼아 성능의 기준점을 설정할 필요가 있습니다.

운용시간, 훈련 월, 가동률 등 다양한 요인과 수요 간의 상관관계를 검정하고 재고 관리상 중요한 핵심 품목을 식별해야 체계적인 부품 수급 관리가 가능합니다.

**핵심 과업**

- Baseline 예측 알고리즘 구현: 3·6개월 이동평균, 단순 지수평활법, 그리고 작년 같은 달 방식을 적용하여 6개월 평가 구간에서의 MAE, WAPE, MAPE 오차를 비교·분석합니다.
- 탐색적 데이터 분석(EDA) 및 상관 검정: 운용시간, 훈련 월, 가동률 등과 부품 수요 간의 상관계수 및 p-value를 검정하고 8종의 EDA 차트를 생성합니다.
- ABC(파레토) 분류 수행: 최근 12개월 수요 금액 기준 상위 20% 품목을 식별하고, 그중 단종 및 단종예고 품목을 파악하여 관리 우선순위를 정합니다.

**검증 및 산출물**

**검증 기준:** EDA 차트 8종 이상 생성, 유의미한 상관 피처 5개 이상 도출($p < 0.05$), Baseline 방식별 오차 비교 및 ABC 상위 품목 중 단종 위험 품목 제시.

**주요 산출물:**

notebooks/mnt02_eda_analysis.ipynb 및 EDA 요약 리포트.

## 📁 이번 미션에서 완성할 것

- **미션에서 완성할 내용**
  - 3·6개월 이동평균, 단순 지수평활, 작년 같은 달 방식의 MAE·WAPE·MAPE를 평가 구간 6개월에서 비교한다.
  - 운용시간·훈련 월·가동률 등과 수요의 상관계수·p-value를 검정하고 EDA 차트 8종을 만든다.
  - ABC(파레토) 분석으로 상위 20% 품목을 식별하고 그중 단종·단종예고 품목을 표시한다.
- **구현 계약**
  - 핵심 구현 계약 src/analysis/mnt02_eda.py / ses: 수요 배열 y,alpha → 한달 앞 SES 예측 배열.
  - 초기값은 첫 관측값이고 이후 예측은 이전 시점의 관측만 사용한다.
  - 빈/비유한/음수 수요와 범위 밖 alpha는 거부한다.
  - 초기 구간은 성능평가에서 제외한다.
- **변경 과제와 검증**
  - 추가 변경·검증 이동평균과 지수평활법으로 1차 부품 소요량 예측 Baseline을 만들고 탐색적 분석으로 수요 요인과 핵심 관리 품목을 찾는다.
  - SES alpha를 0.6으로 바꾼 결과와 최초 후보 선택 결과를 비교하고 차트 8종·상관검정의 해석을 검증한다.
  - 핵심구현 후첫성공실행을보존하고추가변경·재실행·실제AI원문·근거검토3건·보완·최종판단을 작성합니다.

**제출 경로:** `notebooks/mnt02_eda_analysis.ipynb`, `reports/mnt02_eda_summary.md`, `forms/MNT02_M03_forms.md`


---

## 🎯 미션 03. 이동평균 및 지수평활법 기반 1차 부품 소요량 예측 Baseline 알고리즘 구현

**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 이동평균 기반 1차 부품 소요량 예측 Baseline 알고리즘을 개발한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **이동평균과 지수평활법으로 1차 부품 소요량 예측 Baseline을 만들고 탐색적 분석으로 수요 요인과 핵심 관리 품목을 찾는다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **3·6개월 이동평균, 단순 지수평활, 작년 같은 달 방식의 MAE·WAPE·MAPE를 평가 구간 6개월에서 비교 수행하고 검증하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `notebooks/mnt02_eda_analysis.ipynb 및 EDA 요약 리포트` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 의사결정에 직관적인 인사이트 제공 수준

#### 미션 PASS 조건

- 정량 기준: EDA 시각화 차트 8종 이상 생성, 유의미한 상관관계 피처 5개 이상 도출

- 진행 기준: EDA 노트북 실행 완료 및 핵심 발견점(Key Findings) 문서화

- 검증 방법: 통계적 가설 검정(p-value, 상관계수)을 통한 유의성 검증

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


## 📋 순서대로 할 활동

### 활동 1: 계획과 사전 판단

- 원문 목표 분석 및 예측 변수 식별 원본 목표, 기본 제공 코드, 그리고 탐색적 데이터 분석(EDA) 자료를 면밀히 검토하여 어떤 정보가 부품 수요와 상관관계가 있는지 분석합니다.
- 특히 '예측 시점에 실제로 알려져 있는 정보(Available at forecast time)'인지 엄격히 식별하여 미래 시점의 정보가 사전 유출(Data Leakage)되는 오류를 방지하는 계획을 수립합니다.
- AI 위임 범위 및 검증 계획 설정 이동평균(MA)과 지수평활법(SES)을 활용한 1차 소요량 예측 Baseline 구축 과정에서 AI에 위임할 코드 작성 범위와 활용할 자료를 구체적으로 확정합니다.
- 탐색적 데이터 분석을 통해 도출할 수요 요인과 핵심 관리 품목의 정합성을 검증할 구체적인 절차를 사전에 준비합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M03 계획: 이동평균과 지수평활법으로 1차 부품 소요량 예측 Baseline을 만들고 탐색적 분석으로 수요 요인과 핵심 관리 품목을 찾는다.에 필요한 입력·기간·단위·사용 가능 시점과 ses의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2: AI와 코드 변경

- **함수 구현과 첫 실행**
  - 핵심 함수(ses) 구현 및 첫 실행 보존 수요 배열 y와 평활 계수 alpha를 입력받아 한 달 앞 지수평활법(SES) 예측 배열을 반환하는 핵심 함수 ses를 구현하고 저장한 뒤, 첫 성공 실행 결과를 온전히 보존합니다.
  - 초기값은 첫 관측값을 사용하며, 이후 예측은 이전 시점의 관측값만을 활용합니다.
  - 빈 값, 비유한 값(NaN/Inf), 음수 수요 및 유효 범위를 벗어난 alpha값은 철저히 거부하고, 초기 구간은 성능 평가에서 제외합니다.
- **AI 활용과 변경 과제**
  - AI 기획 반영 및 변경 과제 수행 앞서 수립한 예측 기획안을 바탕으로 구현된 코드를 검증하고 AI와 함께 고도화 작업을 진행합니다.
  - 변경 과제: SES의 평활 계수 alpha를 0.6으로 변경한 결과와 최초 후보 선택 결과를 상호 비교하고, 탐색적 데이터 분석(EDA) 차트 8종 및 상관 검정 결과의 통계적 해석을 엄격히 검증합니다.
  - 실제 코드와 실행 출력 결과물은 모두 안전하게 저장합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M03 구현: src/analysis/mnt02_eda.py의 ses를저장하고계약〔수요 배열 y,alpha → 한달 앞 SES 예측 배열.
- 초기값은 첫 관측값이고 이후 예측은 이전 시점의 관측만 사용한다.
- 빈/비유한/음수 수요와 범위 밖 alpha는 거부한다.
- 초기 구간은 성능평가에서 제외한다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3: 근거 대조와 재실행

- 수정 코드 저장 및 재실행 검증 도메인 분석 스크립트에 반영된 수정 코드를 안전하게 저장하고 스크립트를 재실행합니다.
- 오차 분모, 탐색적 데이터 분석(EDA) 차트 8종, ABC 품목별 금액, 그리고 상관계수의 p-value가 실제 출력 결과와 정확히 일치하는지 철저히 대조합니다.
- AI 설명 및 코드 선택 대조 분석 AI가 제시한 설명 및 코드 선택 사항 3건 이상을 실제 실습 자료 및 실행 결과 로그와 상호 대조하여 타당성을 엄격히 검토합니다.
- 보완이 필요한 답변과 수정 사항을 다시 한번 확인하고, MAPE(실제 수요 0 제외 분모)와 WAPE(실제 합계 0일 때 N/A 표시) 산출 기준을 준수하며 상관관계가 곧 인과관계를 의미하지 않음을 명확히 유의합니다.

#### 확인할 사항:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M03 검증: [필수] Baseline·EDA — EDA 차트 8종 이상, 유의미한 상관 피처 5개 이상(p<0.05), Baseline 방식별 오차 비교와 ABC 상위 품목 중 단종 위험 품목을 제시했는가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - MAPE는 실제수요 0을 제외한 분모, WAPE는 실제 합계가 0일 때N/A를 표시합니다.
  - 상관과p-value는 인과가 아닙니다.
  - 가상 생성 구조·반복 품목/월·다중 비교·상관된 피처의 한계를 설명합니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4: 최종 판단과 산출물

- 종합 산출물 제출 및 상태 구분 원본 평가 기준에 따른 최종 결과물인 notebooks/mnt02_eda_analysis.ipynb, reports/mnt02_eda_summary.md, forms/MNT02_M03_forms.md를 누락 없이 체계적으로 제출합니다.
- 미달 항목, 미측정 항목, 문서 미작성 내용, 그리고 실제 승인 대기 상태를 명확히 구분하여 기록합니다.
- 예측 지표 기준 및 한계점 명시 오차 산출 시 MAPE는 실제 수요 0을 제외한 분모를 사용하고, WAPE는 실제 합계가 0일 때 N/A를 표시하는 기준을 엄격히 준수합니다.
- 상관관계(p < 0.05)가 곧 인과관계를 의미하지 않음을 명시하고, 가상 생성 구조, 반복 품목/월 특성, 다중 비교 및 상관된 피처가 지닌 한계점을 철저히 설명합니다.

#### 확인할 사항:

- 이 활동의 확인 근거 M03 완료: notebooks/mnt02_eda_analysis.ipynb, reports/mnt02_eda_summary.md, forms/MNT02_M03_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 이 미션의 AI 활용 예시

- 가상 자료 분석·기획·코드 변경·디버깅·검증 설계·해석
- "교육용 미션 03의 핵심 함수 구현에서 누락되기 쉬운 입력값·경계조건을 알려주세요. 원본 자료에 없는 성능 수치를 만들지 말아 주세요."

AI 답변은 그대로 정답이 아닙니다. **실제 실행 로그와 최소 3건의 주장·선택을 비교**하세요.


## 직접 구현하기

다음 코드의 `TODO`를 완성해 실행합니다. 출력을 확인하지 않았다면 성공했다고 적지 마세요.


In [ ]:
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """이동평균(3·6개월)과 단순 지수평활법으로 품목별 다음 달 부품 소요량을 예측하는 Baseline을 구현하고, 탐색적 분석(EDA) 차트 8종·상관 검정·ABC(파레토) 분류로 수요 특성과 핵심 관리 품목을 도출한다."""
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy import stats

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 야전 정비창 수리부속 청구/출고·장비 가동률 데이터 생성기
    # 실제 부대·품목·재고·단가 정보와 무관하다. 품목 40개 x 48개월(가상 2022-01 ~ 2025-12) 월별 자료를 만든다.
    # ---------------------------------------------------------------------------
    FSC = {"2520": "power_train", "2530": "brake_steer", "2910": "fuel", "2930": "cooling",
           "2940": "filter", "5330": "gasket", "6140": "battery", "5961": "semiconductor"}
    EXERCISE = {3, 8, 10}      # 가상 대규모 훈련 월(훈련 계획으로 미리 알 수 있음)
    N_MONTHS = 48


    def season(fsc, month):
        """품목군별 계절 계수: 축전지는 겨울, 냉각계통은 여름에 수요가 늘어난다."""
        # [TODO : 학생 실습 3] 이동평균 및 지수평활법 기반 1차 부품 소요량 예측 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    def make_items(n=40, seed=2):
        """품목 마스터: NSN(가상), 품목군, 장비 유형, 기준 수요, 단가, 조달 리드타임, 공급업체 수, 생산(단종) 상태."""
        rng = np.random.default_rng(seed)
        rows = []
        for i in range(n):
            f = str(rng.choice(list(FSC)))
            status = str(rng.choice(["생산중", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
            rows.append({"item_id": f"P{i + 1:03d}", "nsn": f"{f}-00-{rng.integers(100, 1000)}-{rng.integers(1000, 10000)}",
                         "fsc": f, "group": FSC[f], "equip_type": str(rng.choice(["A", "B", "C"])),
                         "base": float(np.round(rng.lognormal(2.7, 0.8), 1)),
                         "price": int(rng.lognormal(12, 1.0)) // 100 * 100,
                         "lead_time": int(rng.integers(1, 7) if status == "생산중" else rng.integers(4, 13)),
                         "suppliers": int(rng.integers(1, 5)) if status == "생산중" else 1,
                         "status": status, "alt_available": bool(rng.random() < (0.6 if status == "생산중" else 0.3))})
        return pd.DataFrame(rows)


    def make_history(items, months=N_MONTHS, seed=3):
        """월별 청구 수요(개), 긴급 불출 건수, 장비 유형별 운용시간·가동률."""
        rng = np.random.default_rng(seed)
        t = np.arange(months)
        moy = t % 12 + 1
        ex = np.isin(moy, list(EXERCISE))
        hours = {e: (100 + 20 * rng.random(months)) * np.where(ex, 1.5, 1.0) for e in "ABC"}
        avail = {e: np.clip(0.93 - 0.12 * (hours[e] - 100) / 80 + rng.normal(0, 0.015, months), 0.6, 0.99) for e in "ABC"}
        rows = []
        for it in items.itertuples():
            trend = 1 + rng.uniform(-0.1, 0.3) * t / months
            s = np.array([season(it.fsc, m) for m in moy])
            lam = it.base * s * trend * hours[it.equip_type] / 110
            d = rng.poisson(lam)
            rows.append(pd.DataFrame({"item_id": it.item_id, "t": t, "ym": [f"{2022 + k // 12}-{k % 12 + 1:02d}" for k in t],
                                      "month": moy, "exercise": ex.astype(int), "op_hours": hours[it.equip_type].round(1),
                                      "availability": avail[it.equip_type].round(3), "demand": d,
                                      "emergency": rng.binomial(d, 0.05)}))
        return pd.concat(rows, ignore_index=True)


    def make_stock(items, hist, seed=4):
        """현재(가상 2026-01) 재고: 보유량과 미입고 발주량(조달 중인 물량). 단종 품목은 추가 발주가 어려워 미입고량이 적다."""
        rng = np.random.default_rng(seed)
        recent = hist[hist["t"] >= hist["t"].max() - 5].groupby("item_id")["demand"].mean()
        st = items[["item_id", "lead_time", "status"]].copy()
        st["on_hand"] = [int(round(recent[i] * rng.uniform(0.5, 3.0))) for i in st["item_id"]]
        cover = np.where(st["status"] == "생산중", rng.uniform(0.6, 1.3, len(st)), rng.uniform(0.0, 0.6, len(st)))
        st["on_order"] = [int(round(recent[i] * L * c)) for i, L, c in zip(st["item_id"], st["lead_time"], cover)]
        return st[["item_id", "on_hand", "on_order"]]
    # ---------------------------------------------------------------------------

    items = make_items()
    hist = make_history(items).merge(items[["item_id", "group", "price", "status", "lead_time"]], on="item_id")
    TEST_FROM = 42                          # 마지막 6개월(가상 2025-07 ~ 2025-12)을 평가 구간으로 둔다


    def ses(y, alpha):
        """단순 지수평활: 다음 달 예측 = alpha x 이번 달 실적 + (1 - alpha) x 이번 달 예측."""
        # [TODO : 학생 실습 3] 이동평균 및 지수평활법 기반 1차 부품 소요량 예측 Baseline 알고리즘 구현 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    preds = []
    for iid, g in hist.groupby("item_id"):
        y = g.sort_values("t")["demand"].to_numpy(float)
        best = min([0.1, 0.2, 0.3, 0.5], key=lambda a: np.abs(ses(y, a)[12:TEST_FROM] - y[12:TEST_FROM]).mean())
        s = pd.Series(y)
        preds.append(pd.DataFrame({"item_id": iid, "t": np.arange(len(y)), "y": y, "MA3": s.shift(1).rolling(3).mean(),
                                   "MA6": s.shift(1).rolling(6).mean(), "SES": ses(y, best), "작년 같은 달": s.shift(12)}))
    P = pd.concat(preds)
    T = P[P["t"] >= TEST_FROM]
    ab = items[items["base"] >= 10]["item_id"]                       # 수요가 충분한 품목(MAPE 해석 가능)
    rows = []
    for m in ["MA3", "MA6", "SES", "작년 같은 달"]:
        e = T["y"] - T[m]
        big = T[T["item_id"].isin(ab)]
        rows.append({"방법": m, "MAE(개)": round(e.abs().mean(), 2), "WAPE": f"{e.abs().sum() / T['y'].sum():.1%}",
                     "MAPE(수요 많은 품목)": f"{(np.abs(big['y'] - big[m]) / big['y'].clip(lower=1)).mean():.1%}"})
    base_tbl = pd.DataFrame(rows)
    print("[Baseline 예측 성능 - 평가 구간 6개월]")
    print(base_tbl.to_string(index=False))
    print("해석: 이동평균은 훈련 월 급증과 계절 변화를 한 달 늦게 따라가 오차가 커진다(미션 4에서 개선).")

    # 2) 상관 검정: 품목 평균 대비 수요 비율과 후보 피처
    hist = hist.sort_values(["item_id", "t"])
    hist["ratio"] = hist["demand"] / hist.groupby("item_id")["demand"].transform("mean")
    g = hist.groupby("item_id")["ratio"]
    hist["lag1_ratio"] = g.shift(1); hist["lag12_ratio"] = g.shift(12)
    hist["winter"] = hist["month"].isin([12, 1, 2]).astype(int)
    cand = ["op_hours", "exercise", "availability", "lag12_ratio", "lag1_ratio", "t", "winter", "price"]
    res = []
    for c in cand:
        d = hist[["ratio", c]].dropna()
        r, p = stats.pearsonr(d[c], d["ratio"])
        res.append({"피처": c, "상관계수": round(r, 3), "p-value": f"{p:.1e}", "유의(p<0.05)": "예" if p < 0.05 else "아니오"})
    res = pd.DataFrame(res)
    n_sig = (res["유의(p<0.05)"] == "예").sum()
    print("\n[수요 비율과 피처의 상관 검정]")
    print(res.to_string(index=False))
    print(f"유의한 피처 {n_sig}개 (목표 5개 이상: {'달성' if n_sig >= 5 else '미달'})")
    t_stat, p_ex = stats.ttest_ind(hist[hist["exercise"] == 1]["ratio"], hist[hist["exercise"] == 0]["ratio"])
    print(f"훈련 월과 평시 월 수요 비율 차이 t검정: t={t_stat:.1f}, p={p_ex:.1e}")

    # 3) ABC(파레토) 분류: 최근 12개월 수요 금액 기준
    val = hist[hist["t"] >= 36].groupby("item_id").apply(lambda x: (x["demand"] * x["price"]).sum(), include_groups=False).sort_values(ascending=False)
    cum = val.cumsum() / val.sum()
    abc = pd.Series(np.where(cum <= 0.8, "A", np.where(cum <= 0.95, "B", "C")), index=val.index)
    top = val.index[: int(np.ceil(len(val) * 0.2))]
    print(f"\nABC 분류: A {sum(abc == 'A')}개 / B {sum(abc == 'B')}개 / C {sum(abc == 'C')}개, 상위 20% {len(top)}개 품목이 수요 금액의 {val[top].sum() / val.sum():.0%} 차지")
    st = items.set_index("item_id").loc[top]
    print("상위 20% 중 단종예고·단종 품목:", ", ".join(f"{i}({s})" for i, s in st["status"].items() if s != "생산중") or "없음")

    # 4) EDA 차트 8종
    fig, ax = plt.subplots(2, 4, figsize=(18, 8))
    tot = hist.groupby("t")["demand"].sum()
    ax[0, 0].plot(tot.index, tot.values); ax[0, 0].set_title("1. Total monthly demand")
    hist.groupby(["t", "group"])["demand"].sum().unstack().plot(ax=ax[0, 1], legend=False); ax[0, 1].set_title("2. Demand by FSC group")
    hist.boxplot(column="ratio", by="month", ax=ax[0, 2], grid=False); ax[0, 2].set_title("3. Seasonality (demand ratio)")
    ax[0, 3].scatter(hist["op_hours"], hist["ratio"], s=4, alpha=0.3); ax[0, 3].set_title("4. Operating hours vs demand ratio")
    hist.groupby("exercise")["ratio"].mean().plot.bar(ax=ax[1, 0]); ax[1, 0].set_title("5. Exercise month effect")
    ax[1, 1].plot(np.arange(1, len(cum) + 1), cum.values, "o-", ms=3); ax[1, 1].axhline(0.8, ls="--", c="gray"); ax[1, 1].set_title("6. Pareto curve (ABC)")
    items.assign(st=items["status"].map({"생산중": "active", "단종예고": "EOL notice", "단종": "discontinued"})).boxplot(column="lead_time", by="st", ax=ax[1, 2], grid=False); ax[1, 2].set_title("7. Lead time by production status")
    ag = hist.groupby("t").agg(av=("availability", "mean"), d=("demand", "sum"))
    ax[1, 3].scatter(ag["av"], ag["d"]); ax[1, 3].set_title("8. Availability vs total demand")
    for a in ax.ravel():
        a.set_xlabel(""); a.tick_params(labelsize=7)
    plt.suptitle("MNT02 EDA - 8 charts (synthetic data)"); plt.tight_layout(); plt.show()
    print("EDA 차트 8종 생성 완료. 핵심 발견: 운용시간·훈련 월이 수요를 끌어올리고, 단종 품목은 리드타임이 길어 결품 위험이 크다.")


## 실습 기록과 제출

- 마지막 **통합 산출물·학습기록 입력 화면**을 실행합니다.
- 각 산출물 탭에서 직접 내용을 작성하거나 기존 파일을 불러와 수정하고 개별 저장하세요.
- 학습기록 탭에서 실제 AI 질문·답변, 출력·자료 대조 근거, 검토 3건 이상 및 수정 내용, 최종 판단과 미달·미측정·미실시 상태를 기록합니다.
- 저장 후 ZIP 생성 및 다운로드 버튼을 순서대로 사용하세요.

**통합 입력·저장 기능:** 마지막 통합 화면에서 미션별 산출물을 파일별로 작성·수정·저장하고, 실제 AI 질문·답변 원문, 원자료·출력 대조 근거, 검토 3건, 수정과 최종 수행 상태를 기록한 뒤 ZIP으로 다운로드합니다.

저장만으로 PASS나 승인이 되지 않습니다.


In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('MNT02_PROJECT_READY', False) and globals().get('MNT02_PROJECT_DIR') and Path(MNT02_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 3
        DELIVERABLE_PATHS = ['forms/MNT02_M03_forms.md', 'notebooks/mnt02_eda_analysis.ipynb', 'reports/mnt02_eda_summary.md', 'src/analysis/mnt02_eda.py']
        PROJECT_NAME = 'MNT02'
        NOTEBOOK_ROLE = '학생용 Student'
        WORK_ROOT = Path(MNT02_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'MNT02_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'MNT02_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                state['dirty'] = change['new'] != state['saved_value']
                state['saved'] = state['loaded'] and not state['dirty']
                if state['dirty']:
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
                elif state['saved']:
                    state['status'].value = '<span style="color:#286">저장된 내용과 같습니다.</span>'
                else:
                    state['status'].value = '<span style="color:#777">아직 저장하지 않았습니다.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE, 'notebook_role': NOTEBOOK_ROLE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# MNT02 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if record_dirty['value'] or any(v['dirty'] for v in editors.values()):
                    print('미저장 변경이 있습니다. 먼저 저장 후 ZIP을 다시 만드세요.')
                    return
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ MNT02 미션 {MISSION_CODE:02d} ({NOTEBOOK_ROLE}) 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
